# Предсказание количества атомов

In [1]:
# Импорт необходимых библиотек
from src.pdb.PDBFile import PDBFile
from tqdm import tqdm
from src.preprocessing.interpolation import resample_trajectory, resample_trajectory_cubic_spline
from src.preprocessing.curves import get_invariant_features
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import torch
import torch.nn.functional as F
import warnings
import os
from sklearn.model_selection import train_test_split
from src.utils.config_loader import project_config
warnings.filterwarnings('ignore')

In [2]:
# Загрузка данных
K = 2048
CA_POINT_ARRAY = []
import os
zero_error_count = 0
for root, dirs, files in os.walk('data/new_dompdb/dompdb'):
    for file in tqdm(files):
        file_path = os.path.join(root, file)
        pdb_file = PDBFile(file_path)
        if len(pdb_file.atom_line) > 0:
            ca_point = pdb_file.get_point_cloud()
            CA_POINT_ARRAY.append(ca_point)
        else: 
            zero_error_count += 1

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 119369/119369 [04:50<00:00, 410.64it/s]


In [3]:
# Предобработка данных
resampled_coords = []
invariand_ca_coords = []
failed_indices = []  

for idx, example in enumerate(tqdm(CA_POINT_ARRAY)):
    try:
        invariant_features = get_invariant_features(np.array(example))
        
        if len(invariant_features) < 4:
            print(f"Пример {idx}: недостаточно точек ({len(invariant_features)}) для сплайна, пропускаем")
            failed_indices.append((idx, "Недостаточно точек"))
            continue
            
        resampled = resample_trajectory_cubic_spline(invariant_features, k=2048)
        
        if resampled is not None and len(resampled) == 2048:
            resampled_coords.append(resampled)
            invariand_ca_coords.append(invariant_features)
        else:
            print(f"Пример {idx}: некорректный результат ресемплинга")
            failed_indices.append((idx, "Некорректный результат"))
            
    except ValueError as e:
        print(f"Пример {idx}: ValueError - {e}")
        failed_indices.append((idx, f"ValueError: {e}"))
    except Exception as e:
        print(f"Пример {idx}: неожиданная ошибка - {type(e).__name__}: {e}")
        failed_indices.append((idx, f"{type(e).__name__}: {e}"))

 38%|█████████████████████████████████████████████████████████████████████████▎                                                                                                                       | 45358/119369 [06:54<11:01, 111.80it/s]

Пример 45342: ValueError - x_i должны быть строго возрастающими


 41%|███████████████████████████████████████████████████████████████████████████████▋                                                                                                                 | 49292/119369 [07:30<10:35, 110.25it/s]

Пример 49273: ValueError - x_i должны быть строго возрастающими


 72%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊                                                     | 86443/119369 [13:08<04:54, 111.71it/s]

Пример 86425: ValueError - x_i должны быть строго возрастающими


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 119369/119369 [18:09<00:00, 109.61it/s]


In [4]:
invariand_ca_coords[0].shape, resampled_coords[0].shape

((134, 3), (2048, 3))

In [5]:
import numpy as np

masks = []

for idx in tqdm(range(len(resampled_coords))):
    original_coords = invariand_ca_coords[idx]  
    resampled = resampled_coords[idx]  
    
    mask = np.zeros(2048, dtype=int)
    
    for i, orig_point in enumerate(original_coords):
        distances = np.linalg.norm(resampled - orig_point, axis=1)
        closest_idx = np.argmin(distances)
        mask[closest_idx] = 1
    
    masks.append(mask)


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 119366/119366 [11:39<00:00, 170.62it/s]


In [10]:
class CurvesToAtomDataset(Dataset):
    def __init__(self, curves, mask):
        self.data = curves
        self.mask = mask
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        curve = self.data[idx]
        mask = self.mask[idx]
        return torch.tensor(curve), torch.tensor(mask) 

In [11]:
config = project_config.load()  
train_X, val_X, train_Y, val_Y = train_test_split(
    resampled_coords,
    masks,
    test_size=config.get('training', {}).get('validation_split', 0.1),  
    random_state=42 
)

In [12]:
train_dataset = CurvesToAtomDataset(train_X, train_Y)

In [22]:
val_dataset = CurvesToAtomDataset(val_X, val_Y)

In [15]:
train_dataset[0][0].shape,train_dataset[0][1].shape

(torch.Size([2048, 3]), torch.Size([2048]))

In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F
class Conv1DStack(nn.Module):
    def __init__(self):
        super().__init__()
        self.convs = nn.Sequential(
            nn.Conv1d(3, 64, 7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Conv1d(64, 128, 5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Conv1d(128, 256, 3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Conv1d(256, 1, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.convs(x.transpose(1, 2)).squeeze(1)

In [66]:
class WeightedBCELoss(nn.Module):
    def __init__(self, pos_weight=1.0):
        super().__init__()
        self.pos_weight = pos_weight 
    
    def forward(self, predictions, targets):

        batch_size = targets.shape[0]
        total_elements = targets.numel()
        
        num_pos = torch.sum(targets > 0.5).float()
        num_neg = total_elements - num_pos
        
        weight_pos = num_neg / (num_pos + 1e-8)
        weight_neg = num_pos / (num_neg + 1e-8)
        
        weights = targets * weight_pos + (1 - targets) * weight_neg
        weights = weights * self.pos_weight  # Дополнительный вес для атомов
        bce = F.binary_cross_entropy(predictions, targets, reduction='none')
        weighted_bce = (bce * weights).mean()
        return weighted_bce

In [118]:
batch_size = 16
learning_rate = 0.001

In [119]:
train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )

In [120]:
val_loader = DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False
        )

In [121]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Conv1DStack()

In [122]:
criterion = WeightedBCELoss(pos_weight=5.0)

In [123]:
optimizer = optim.Adam( 
        model.parameters(),
        lr=learning_rate,
    )

In [124]:
epochs = 30

In [125]:
model = model.to(device)

history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    
    for curves, masks in tqdm(train_loader):
        # Convert to float32 and move to device
        curves = curves.float().to(device)
        masks = masks.float().to(device)
        
        outputs = model(curves)
        loss = criterion(outputs, masks)
        
        optimizer.zero_grad()
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)
    history['train_loss'].append(avg_train_loss)
    
    if val_dataset:
        model.eval()
        val_loss = 0.0
        val_accuracy = 0.0
        
        with torch.no_grad():
            for curves, masks in val_loader:
                curves = curves.float().to(device)
                masks = masks.float().to(device)
                
                outputs = model(curves)
                loss = criterion(outputs, masks)
                val_loss += loss.item()
                
                preds = (outputs > 0.5).float()
                accuracy = (preds == masks).float().mean()
                val_accuracy += accuracy.item()
        
        avg_val_loss = val_loss / len(val_loader)
        avg_val_acc = val_accuracy / len(val_loader)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(avg_val_acc)
    
    print(f"Epoch {epoch+1:3d}/{epochs} | "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | "
          f"Val Acc: {avg_val_acc:.4f}")

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6715/6715 [00:55<00:00, 121.77it/s]


Epoch   1/30 | Train Loss: 0.7047 | Val Loss: 0.5068 | Val Acc: 0.7132


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6715/6715 [00:55<00:00, 121.84it/s]


Epoch   2/30 | Train Loss: 0.4015 | Val Loss: 0.3979 | Val Acc: 0.7087


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6715/6715 [00:55<00:00, 121.33it/s]


Epoch   3/30 | Train Loss: 0.2886 | Val Loss: 0.7543 | Val Acc: 0.9274


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6715/6715 [00:55<00:00, 121.06it/s]


Epoch   4/30 | Train Loss: 0.2373 | Val Loss: 0.5250 | Val Acc: 0.8213


 99%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋  | 6637/6715 [00:54<00:00, 120.85it/s]


KeyboardInterrupt: 

In [116]:
(model(val_dataset[0][0].unsqueeze(0).to(device).float()) > 0.5).sum()

tensor(174, device='cuda:0')

In [117]:
val_dataset[0][1].sum()

tensor(87)